Healthcare Analytics Project

In [84]:
print("Dataset Loaded\n")
df = pd.read_csv(r"C:\Users\navee\OneDrive\Desktop\HR ANALYTICS\US Healthcare dataset\us_healthcare_raw.csv")

Dataset Loaded



In [86]:
print("Libraries Imported\n")
import pandas as pd
import numpy as np

Libraries Imported



DATA EXPLORATION

In [96]:
print(f"Rows & Columns:\n{df.shape}")

Rows & Columns:
(20500, 9)


In [112]:
print("Columns Types:\n")
df.info()

Columns Types:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20500 entries, 0 to 20499
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   claim_id        20500 non-null  int64  
 1   patient_id      20500 non-null  int64  
 2   service_date    20500 non-null  object 
 3   provider_name   16405 non-null  object 
 4   diagnosis       16398 non-null  object 
 5   procedure       16348 non-null  object 
 6   charge_amount   19998 non-null  float64
 7   payment_amount  19994 non-null  float64
 8   insurance_type  15358 non-null  object 
dtypes: float64(2), int64(2), object(5)
memory usage: 1.4+ MB


In [122]:
print(f"Columns Names:\n {df.columns.tolist()}")

Columns Names:
 ['claim_id', 'patient_id', 'service_date', 'provider_name', 'diagnosis', 'procedure', 'charge_amount', 'payment_amount', 'insurance_type']


In [131]:
print("Nulls in each columns")
df.isnull().sum()

Nulls in each columns


claim_id             0
patient_id           0
service_date         0
provider_name     4095
diagnosis         4102
procedure         4152
charge_amount      502
payment_amount     506
insurance_type    5142
dtype: int64

In [148]:
print("Duplicates in dataset")
df[df.duplicated()]

Duplicates in dataset


,claim_id,patient_id,service_date,provider_name,diagnosis,procedure,charge_amount,payment_amount,insurance_type
20000,10602,1585,2022-08-12,NaN,Hypertension,CT Scan,4654.11,4388.768196,Medicaid
20001,11351,1061,2024-02-04,Clinic X,Diabetes,MRI,3870.32,3643.940529,Medicare
20002,2468,4860,2020-02-04,NaN,Diabetes,MRI,4728.23,4719.758345,Medicare
20003,16335,2727,2020-02-22,Clinic Y,NaN,MRI,3316.30,2969.277130,Medicare
20004,17212,3421,2021-02-27,NaN,Covid,MRI,820.64,484.925309,Private
...,...,...,...,...,...,...,...,...,...
20495,6707,1382,2022-12-07,Clinic Y,NaN,CT Scan,996.34,912.764222,Private
20496,18075,4003,2022-06-11,Hosp B,NaN,NaN,1294.32,1171.022548,NaN
20497,16621,3780,2021-06-12,Hosp A,Covid,NaN,4659.40,4255.426992,Medicare
20498,17362,1333,2021-09-22,Clinic Y,Diabetes,MRI,3391.39,3350.141463,Medicare


DATA CLEANING 

In [160]:
print("Duplicates Removed")
df = df.drop_duplicates(subset='claim_id')

Duplicates Removed


FIXING MISSING VALUES

In [175]:
df.loc[:,'provider_name'] = df['provider_name'].fillna('Unknown Provider')
df.loc[:,'diagnosis'] = df['diagnosis'].fillna('Unknown Diagnosis')
df.loc[:,'procedure'] = df['procedure'].fillna('Not Specified')
df.loc[:,'insurance_type'] = df['insurance_type'].fillna('Self Pay')
df.loc[:,'charge_amount'] = df['charge_amount'].fillna(df['charge_amount'].mean())
df.loc[:,'payment_amount'] = df['payment_amount'].fillna(0)

STANDARDIZATION COLUMNS

In [187]:
df.loc[:,'provider_name'] = df['provider_name'].str.strip().str.title()
df.loc[:,'diagnosis'] =  df['diagnosis'].str.strip().str.title()
df.loc[:,'procedure'] = df['procedure'].str.strip().str.title()
df.loc[:,'insurance_type'] = df['insurance_type'].str.strip().str.title()

FIXING DATE

In [193]:
df.loc[:,'service_date'] = pd.to_datetime(df['service_date'])

FEATURE ENGINEERING

In [210]:
df['profit'] = df['charge_amount'] - df['payment_amount']
df['year'] = df['service_date'].dt.year
df['month'] = df['service_date'].dt.month

HANDLING OUTLIER

In [213]:
Q1 = df['payment_amount'].quantile(0.25)
Q3 = df['payment_amount'].quantile(0.75)
IQR = Q3 -Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

df = df[(df['payment_amount'] >= lower) & (df['payment_amount'] <= upper)]

In [215]:
df['profit_flag'] = np.where(df['profit'] > 0, 'Profit', 'Loss')

EXPORTING DATASET

In [224]:
print("Data cleaning completed successfully")
df.to_csv(r"C:\Users\navee\OneDrive\Desktop\HR ANALYTICS\US Healthcare dataset/healthcare_cleaned.csv", index=False)

Data cleaning completed successfully


EDA

In [233]:
#Total Revenue by Insurance Type
df.groupby('insurance_type')['charge_amount'].sum().sort_values(ascending=False)

insurance_type
Medicare    1.282254e+07
Private     1.276841e+07
Self Pay    1.276132e+07
Medicaid    1.273490e+07
Name: charge_amount, dtype: float64

In [235]:
#Top 5 Diagnosis
df.groupby('diagnosis').value_counts().head(5)

diagnosis  claim_id  patient_id  service_date  provider_name     procedure      charge_amount  payment_amount  insurance_type  profit      year  month  profit_flag
Covid      16        4444        2020-04-16    Clinic Y          Ct Scan        4274.41        3982.702096     Medicare        291.707904  2020  4      Profit         1
           18        3919        2023-07-09    Unknown Provider  X-Ray          4257.87        3812.551956     Medicare        445.318044  2023  7      Profit         1
           23        1769        2020-06-03    Clinic Y          Mri            1788.57        1519.133391     Private         269.436609  2020  6      Profit         1
           24        3391        2022-11-02    Hosp B            Ct Scan        1931.33        1762.153700     Medicare        169.176300  2022  11     Profit         1
           27        3853        2022-09-19    Unknown Provider  Not Specified  1388.24        1359.601147     Self Pay        28.638853   2022  9      Profit  

In [250]:
#Top Providers by Revenue
df.groupby('provider_name')['charge_amount'].sum().sort_values(ascending=False)

provider_name
Clinic Y            1.049895e+07
Hosp B              1.034300e+07
Hosp A              1.012814e+07
Unknown Provider    1.012582e+07
Clinic X            9.991257e+06
Name: charge_amount, dtype: float64

In [257]:
#Monthly Trend
df.groupby('month')['claim_id'].count()

month
1     2042
2     1576
3     1684
4     1572
5     1646
6     1636
7     1673
8     1647
9     1653
10    1595
11    1622
12    1654
Name: claim_id, dtype: int64

In [275]:
#Average Payment by Insurance
df.groupby('insurance_type')['payment_amount'].mean().round(2)

insurance_type
Medicaid    2247.43
Medicare    2268.97
Private     2250.18
Self Pay    2217.64
Name: payment_amount, dtype: float64